# Prefix-Conditioned Target Suffix Scoring（Teacher-Forced，弱 Recovery）

本 notebook 目标：
- 在不修改仓库源码前提下，基于现有 checkpoint 做 prefix-conditioned suffix scoring。
- 只做 teacher forcing（弱 recovery），不做 generate/sample 的强 recovery。
- 比较 clean / corrupted / patched 三种条件。

checkpoint 路径使用固定服务器路径：
- /scratch/tz6g22/FINAL_report_project/mini_gpt_attnres_runs/tinystories_dual/standard/ckpt_standard_last.pt
- /scratch/tz6g22/FINAL_report_project/mini_gpt_attnres_runs/tinystories_dual/attnres/ckpt_attnres_last.pt


In [ ]:
import json
import sys
from copy import deepcopy
from pathlib import Path

import torch
import matplotlib.pyplot as plt

# 定位仓库根目录
CANDIDATES = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = None
for p in CANDIDATES:
    if (p / 'mini_gpt_attnres').exists():
        REPO_ROOT = p
        break
if REPO_ROOT is None:
    raise RuntimeError('未找到仓库根目录（应包含 mini_gpt_attnres 目录）')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from mini_gpt_attnres.evaluate import load_checkpoint
from mini_gpt_attnres.config import DataConfig
from mini_gpt_attnres.data import build_dataloaders
from mini_gpt_attnres.interp.patching import patch_from_cache

print('REPO_ROOT:', REPO_ROOT)
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)


In [ ]:
# 固定使用服务器上的两个 checkpoint 路径（不扫描本地目录）
device = torch.device('cpu')
CHECKPOINTS = {
    'standard': '/scratch/tz6g22/FINAL_report_project/mini_gpt_attnres_runs/tinystories_dual/standard/ckpt_standard_last.pt',
    'attnres': '/scratch/tz6g22/FINAL_report_project/mini_gpt_attnres_runs/tinystories_dual/attnres/ckpt_attnres_last.pt',
}

# 在这里切换当前实验使用的模型：'standard' 或 'attnres'
MODEL_VARIANT = 'standard'
if MODEL_VARIANT not in CHECKPOINTS:
    raise ValueError(f"MODEL_VARIANT 必须是 {list(CHECKPOINTS.keys())}")

ckpt_path = Path(CHECKPOINTS[MODEL_VARIANT])
model, experiment, checkpoint = load_checkpoint(ckpt_path, device=device)
model = model.to(device).eval()

print('model_variant:', MODEL_VARIANT)
print('checkpoint:', ckpt_path)
print('model_type:', checkpoint.get('model_type'))
print('step:', checkpoint.get('step'))
print('n_layer:', experiment.model.n_layer)
print('n_head:', experiment.model.n_head)
print('n_embd:', experiment.model.n_embd)
print('vocab_size:', experiment.model.vocab_size)
print('block_size:', experiment.model.block_size)
print('device:', next(model.parameters()).device)


In [ ]:
# 样本来源：优先 retrieval，失败时退回 repeated_pattern
sample_model_config = deepcopy(experiment.model)
sample_model_config.block_size = int(min(32, experiment.model.block_size))

dataset_source = None
fallback_reason = None

try:
    retrieval_cfg = DataConfig(
        dataset_type='retrieval',
        train_size=16,
        val_size=8,
        retrieval_pairs=4,
        block_stride=max(1, sample_model_config.block_size),
    )
    train_loader, _ = build_dataloaders(
        model_config=sample_model_config,
        data_config=retrieval_cfg,
        batch_size=4,
        num_workers=0,
        seed=123,
        distributed=False,
    )
    inputs, targets = next(iter(train_loader))
    dataset_source = 'retrieval'
except Exception as e:
    fallback_reason = repr(e)
    repeated_cfg = DataConfig(
        dataset_type='repeated_pattern',
        train_size=16,
        val_size=8,
        pattern_length=8,
        block_stride=max(1, sample_model_config.block_size),
    )
    train_loader, _ = build_dataloaders(
        model_config=sample_model_config,
        data_config=repeated_cfg,
        batch_size=4,
        num_workers=0,
        seed=123,
        distributed=False,
    )
    inputs, targets = next(iter(train_loader))
    dataset_source = 'repeated_pattern'

# 从 (inputs, targets) 重构完整序列 z=[prefix||suffix]，z 长度 = block_size + 1
full_clean = torch.cat([inputs, targets[:, -1:].clone()], dim=1).to(device=device, dtype=torch.long)

full_len = full_clean.size(1)
prefix_len = max(2, full_len // 2)
if prefix_len >= full_len:
    prefix_len = full_len - 1
suffix_len = full_len - prefix_len
if suffix_len < 1:
    raise RuntimeError('suffix 长度必须 >= 1')

# corrupted: 只改 prefix 区域的一个 token（不改 suffix）
full_corrupted = full_clean.clone()
corrupt_index = min(max(1, prefix_len // 2), prefix_len - 1)
full_corrupted[:, corrupt_index] = (full_corrupted[:, corrupt_index] + 1) % experiment.model.vocab_size

clean_input_ids = full_clean[:, :-1]
corrupted_input_ids = full_corrupted[:, :-1]

print('dataset_source:', dataset_source)
if fallback_reason is not None:
    print('fallback_reason:', fallback_reason)
print('batch_size:', full_clean.size(0), 'full_len:', full_len)
print('prefix_len:', prefix_len, 'suffix_len:', suffix_len)
print('suffix starts at token index:', prefix_len, '(in full sequence z)')
print('clean[0] prefix:', full_clean[0, :prefix_len].tolist())
print('clean[0] suffix:', full_clean[0, prefix_len:].tolist())
print('corrupted[0] prefix:', full_corrupted[0, :prefix_len].tolist())


In [ ]:
# 基础 scoring helper（只写 notebook 内 glue code）
def suffix_metrics_from_logits(logits: torch.Tensor, target_full_tokens: torch.Tensor, prefix_len: int, topk: int = 5):
    # logits: [B, L-1, V], target_full_tokens: [B, L]
    targets = target_full_tokens[:, 1:]  # [B, L-1]
    start = prefix_len - 1               # logits/targets 上的 suffix 起点
    if start < 0 or start >= targets.size(1):
        raise ValueError(f'非法 prefix_len={prefix_len} for targets_len={targets.size(1)}')

    suffix_targets = targets[:, start:]   # [B, S]
    suffix_logits = logits[:, start:, :]  # [B, S, V]

    log_probs = torch.log_softmax(suffix_logits, dim=-1)
    per_pos_logprob = torch.gather(log_probs, -1, suffix_targets.unsqueeze(-1)).squeeze(-1)  # [B, S]

    suffix_sum_logprob = per_pos_logprob.sum(dim=1)
    suffix_avg_logprob = per_pos_logprob.mean(dim=1)

    pred_next = suffix_logits.argmax(dim=-1)
    exact_next_token_hit = (pred_next == suffix_targets).float().mean(dim=1)

    k = int(min(topk, suffix_logits.size(-1)))
    topk_ids = torch.topk(suffix_logits, k=k, dim=-1).indices
    topk_hit = (topk_ids == suffix_targets.unsqueeze(-1)).any(dim=-1).float().mean(dim=1)

    target_logits = torch.gather(suffix_logits, -1, suffix_targets.unsqueeze(-1))
    target_rank = (suffix_logits > target_logits).sum(dim=-1) + 1

    return {
        'suffix_sum_logprob_per_sample': suffix_sum_logprob,
        'suffix_avg_logprob_per_sample': suffix_avg_logprob,
        'per_position_suffix_token_logprob': per_pos_logprob,
        'exact_next_token_hit_per_sample': exact_next_token_hit,
        'topk_hit_per_sample': topk_hit,
        'target_rank': target_rank,
        'suffix_sum_logprob': float(suffix_sum_logprob.mean().item()),
        'suffix_avg_logprob': float(suffix_avg_logprob.mean().item()),
        'exact_next_token_hit': float(exact_next_token_hit.mean().item()),
        'topk_hit': float(topk_hit.mean().item()),
        'per_position_curve': per_pos_logprob.mean(dim=0).detach().cpu(),
    }

def pretty_metrics(name: str, m: dict):
    print(f"[{name}] suffix_sum_logprob = {m['suffix_sum_logprob']:.6f}")
    print(f"[{name}] suffix_avg_logprob = {m['suffix_avg_logprob']:.6f}")
    print(f"[{name}] exact_next_token_hit = {m['exact_next_token_hit']:.4f}")
    print(f"[{name}] topk_hit = {m['topk_hit']:.4f}")


In [ ]:
# clean 条件
with torch.no_grad():
    clean_outputs = model(clean_input_ids, return_cache=True, return_intermediates=True)

clean_metrics = suffix_metrics_from_logits(
    logits=clean_outputs['logits'],
    target_full_tokens=full_clean,
    prefix_len=prefix_len,
    topk=5,
)

pretty_metrics('clean', clean_metrics)
print('[clean] per-position curve length:', len(clean_metrics['per_position_curve']))


In [ ]:
# corrupted 条件
with torch.no_grad():
    corrupted_outputs = model(corrupted_input_ids, return_cache=True, return_intermediates=True)

# 注意：仍用 clean 的 suffix 作为 target，衡量恢复能力
corrupted_metrics = suffix_metrics_from_logits(
    logits=corrupted_outputs['logits'],
    target_full_tokens=full_clean,
    prefix_len=prefix_len,
    topk=5,
)

pretty_metrics('corrupted', corrupted_metrics)

print('\n[delta corrupted-clean] suffix_avg_logprob =', corrupted_metrics['suffix_avg_logprob'] - clean_metrics['suffix_avg_logprob'])
print('[delta corrupted-clean] exact_hit =', corrupted_metrics['exact_next_token_hit'] - clean_metrics['exact_next_token_hit'])
print('[delta corrupted-clean] topk_hit =', corrupted_metrics['topk_hit'] - clean_metrics['topk_hit'])


In [ ]:
# patched 条件：复用现有 patching/cache/override 链路
clean_cache = clean_outputs['cache']
cache_keys = sorted(clean_cache.data.keys())
patch_candidates = [k for k in cache_keys if k.endswith(('attn_out', 'mlp_out', 'output'))]
if not patch_candidates:
    raise RuntimeError('未找到可 patch 的 site。')

preferred_site = 'blocks.0.attn_out'
patch_site = preferred_site if preferred_site in patch_candidates else patch_candidates[0]
overrides = patch_from_cache(clean_cache, patch_site)

with torch.no_grad():
    patched_outputs = model(
        corrupted_input_ids,
        return_cache=True,
        return_intermediates=True,
        activation_overrides=overrides,
    )

patched_metrics = suffix_metrics_from_logits(
    logits=patched_outputs['logits'],
    target_full_tokens=full_clean,
    prefix_len=prefix_len,
    topk=5,
)

print('patch_site:', patch_site)
pretty_metrics('patched', patched_metrics)

deltas = {
    'delta_corrupted_minus_clean': {
        'suffix_avg_logprob': corrupted_metrics['suffix_avg_logprob'] - clean_metrics['suffix_avg_logprob'],
        'exact_hit': corrupted_metrics['exact_next_token_hit'] - clean_metrics['exact_next_token_hit'],
        'topk_hit': corrupted_metrics['topk_hit'] - clean_metrics['topk_hit'],
    },
    'delta_patched_minus_corrupted': {
        'suffix_avg_logprob': patched_metrics['suffix_avg_logprob'] - corrupted_metrics['suffix_avg_logprob'],
        'exact_hit': patched_metrics['exact_next_token_hit'] - corrupted_metrics['exact_next_token_hit'],
        'topk_hit': patched_metrics['topk_hit'] - corrupted_metrics['topk_hit'],
    },
}
print(json.dumps(deltas, indent=2, ensure_ascii=False))


In [ ]:
# 可视化
conditions = ['clean', 'corrupted', 'patched']
avg_values = [
    clean_metrics['suffix_avg_logprob'],
    corrupted_metrics['suffix_avg_logprob'],
    patched_metrics['suffix_avg_logprob'],
]

curves = {
    'clean': clean_metrics['per_position_curve'],
    'corrupted': corrupted_metrics['per_position_curve'],
    'patched': patched_metrics['per_position_curve'],
}

hit_exact = [
    clean_metrics['exact_next_token_hit'],
    corrupted_metrics['exact_next_token_hit'],
    patched_metrics['exact_next_token_hit'],
]
hit_topk = [
    clean_metrics['topk_hit'],
    corrupted_metrics['topk_hit'],
    patched_metrics['topk_hit'],
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1) suffix avg logprob 柱状图
axes[0].bar(conditions, avg_values)
axes[0].set_title('Suffix Avg Logprob')
axes[0].set_ylabel('logprob')

# 2) per-position suffix token logprob 曲线图
x = list(range(len(curves['clean'])))
for name, curve in curves.items():
    axes[1].plot(x, curve.tolist(), marker='o', label=name)
axes[1].set_title('Per-position Suffix Token Logprob')
axes[1].set_xlabel('suffix position index')
axes[1].set_ylabel('logprob')
axes[1].legend()

# 3) exact hit / top-k hit 对照图
bar_x = list(range(len(conditions)))
w = 0.35
axes[2].bar([i - w/2 for i in bar_x], hit_exact, width=w, label='exact_hit')
axes[2].bar([i + w/2 for i in bar_x], hit_topk, width=w, label='topk_hit')
axes[2].set_xticks(bar_x)
axes[2].set_xticklabels(conditions)
axes[2].set_ylim(0.0, 1.0)
axes[2].set_title('Hit Rates')
axes[2].legend()

plt.tight_layout()
plt.show()

summary_table = [
    {'condition': 'clean', 'suffix_avg_logprob': clean_metrics['suffix_avg_logprob'], 'exact_hit': clean_metrics['exact_next_token_hit'], 'topk_hit': clean_metrics['topk_hit']},
    {'condition': 'corrupted', 'suffix_avg_logprob': corrupted_metrics['suffix_avg_logprob'], 'exact_hit': corrupted_metrics['exact_next_token_hit'], 'topk_hit': corrupted_metrics['topk_hit']},
    {'condition': 'patched', 'suffix_avg_logprob': patched_metrics['suffix_avg_logprob'], 'exact_hit': patched_metrics['exact_next_token_hit'], 'topk_hit': patched_metrics['topk_hit']},
]
print(json.dumps(summary_table, indent=2, ensure_ascii=False))


## 结果说明

本 notebook 已完成：
1. 用 checkpoint 恢复模型并在 `eval` 下前向。
2. 构造 `z=[prefix||suffix]`，按 teacher forcing 计算 suffix 区间指标。
3. 比较 clean / corrupted / patched 三个条件。

这属于 Prefix-conditioned suffix scoring，因为目标是：
- 固定 prefix 条件下，衡量 target suffix token 的概率恢复能力。
- 指标全部来自 logits + targets 的 teacher-forced 计算。

这属于弱 recovery，而非完整 extraction：
- 未做逐 token generate/sample 攻击；
- 仅评估在目标 suffix 上的打分与命中率恢复。

当前结果可用于快速判断：
- corrupted 是否降低 suffix 恢复分数；
- patched 是否在所选 site 上带来可测恢复增益。
